## 0. Setup

#### A. Install

In [3]:
!pip install --quiet --upgrade pip
!pip install --quiet hf_xet accelerate bitsandbytes datasets fast-langdetect fastexcel Levenshtein lingua-language-detector lxml matplotlib networkx numpy openpyxl optuna pandas polars pyarrow scikit-learn scipy seaborn sentencepiece torch tqdm transformers ipywidgets boto3 altair more-itertools frozendict statsmodels pymc umap-learn
!pip install --quiet -e /home/ec2-user/SageMaker/mse/

ERROR: /home/ec2-user/SageMaker/mse/ is not a valid editable requirement. It should either be a path to a local project or a VCS URL (beginning with bzr+http, bzr+https, bzr+ssh, bzr+sftp, bzr+ftp, bzr+lp, bzr+file, git+http, git+https, git+ssh, git+git, git+file, hg+file, hg+http, hg+https, hg+ssh, hg+static-http, svn+ssh, svn+http, svn+https, svn+svn, svn+file).


#### B. Packages

In [4]:
from __future__ import annotations

# Standard library imports
import json
import os
from collections import defaultdict
from itertools import pairwise

# Third-party imports
import altair as alt
import networkx as nx
import numpy as np
import optuna
import pandas as pd
import polars as pl
import regex
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import torch
from optuna import Trial
from tqdm.auto import tqdm
from matplotlib.ticker import FuncFormatter
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

# Local imports
from mse.cdp.exploration import Exploration
from mse.cdp.merging import Merger
from mse.cdp.processing import Processor
from mse.cdp.classification import Classifier
from mse.cdp.similarity import SimilarityModel
from mse.cdp.translation import Translator
from mse.utils.io import read_csv, read_excel, read_parquet, write_csv, write_parquet
from mse.utils.cdp import DataFrame, Workbook
from mse.utils.aws import shut_down_current_sagemaker_instance

#### C. Environment

In [5]:
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
os.environ['TOKENIZERS_PARALLELISM'] = 'true'

#### D. Globals

In [6]:
class GLOBALS:

    class inputs:
        labels              = ['investor', 'supply_chain']
        years               = [2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020]
    
    class paths:

        # | Local
        # |-------------------
        base                = '/Users/chase/Desktop/code/projects/mse/data/cdp/output'

        configs             = f'{base}/configs.json'

        raw                 = f'{base}/raw/{{label}}/{{year}}.xlsx'

        eda_sheets          = f'{base}/eda/sheets.csv'
        eda_columns         = f'{base}/eda/columns.csv'
        eda_redundant       = f'{base}/eda/redundant.csv'

        processed           = f'{base}/processed/{{label}}/{{year}}.parquet'

        mapping             = f'{base}/mapping/manual/mapping.csv'
        mapping_truth       = f'{base}/mapping/manual/mapping.csv'
        mapping_coalesced   = f'{base}/mapping/manual/coalesced.csv'

        merged              = f'{base}/merged/manual/merged.parquet'
        translated          = f'{base}/merged/manual/translated.parquet'
        classified          = f'{base}/merged/manual/classified.parquet'
        classified_chunk    = f'{base}/merged/manual/classified_{{}}.parquet'
        metrics             = f'{base}/merged/manual/metrics.parquet'
        
        # | SageMaker
        # |-------------------
        # base                = 's3://mse2024'

        # merged              = f'{base}/merged.parquet'
        # translated          = f'{base}/translated.parquet'
        # classified          = f'{base}/classified.parquet'
        # classified_chunk    = f'{base}/classified_{{}}.parquet'
        # metrics             = f'{base}/metrics.parquet'

    class models:
        eda = Exploration(
            min=2
        )
        similarity = SimilarityModel(
            model_name='sentence-transformers/all-MiniLM-L6-v2' # 'all-mpnet-base-v2'
        )
        merger = Merger(
            model=similarity,
            w_D_col=1.7,
            w_S_col=1.8,
            w_S_field=0.3,
            w_S_context=1.8,
            n_fields=4,
            win_S_context=1,
            thresh_C_method='fixed',
            thresh_C=0.24
        )
        processor = Processor(
            merger=merger,
            column_property_join = '=',
            column_property_separator = '|',
            column_suffix_special = '*'
        )
        translator = Translator(
            target='en',
            model_name='alirezamsh/small100'
        )
        classifier = Classifier(
            models=[
                ('climate_specificity', 'climatebert/distilroberta-base-climate-specificity', 'climatebert/distilroberta-base-climate-specificity'),
                ('environmental_claims', 'climatebert/environmental-claims', 'climatebert/environmental-claims'),
                ('transition', 'climatebert/transition-physical', 'climatebert/distilroberta-base-climate-detector'),
                ('climate_commitment', 'climatebert/distilroberta-base-climate-commitment', 'climatebert/distilroberta-base-climate-commitment'),
                ('tcfd', 'climatebert/distilroberta-base-climate-tcfd', 'climatebert/distilroberta-base-climate-tcfd'),
                ('climate_sentiment', 'climatebert/distilroberta-base-climate-sentiment', 'climatebert/distilroberta-base-climate-sentiment'),
                ('netzero_reduction', 'climatebert/netzero-reduction', 'climatebert/distilroberta-base-climate-f'),
                ('renewable', 'climatebert/renewable', 'climatebert/distilroberta-base-climate-detector'),
            ]
        )

    class labels:
        climate_specificity_yes = 'spec'
        climate_specificity_no = 'non'

        environmental_claims_yes = 'yes'
        environmental_claims_no = 'no'

        transition_yes = 'LABEL_1'
        transition_no = 'LABEL_0'

        climate_commitment_yes = 'yes'
        climate_commitment_no = 'no'

        tcfd_strategy = 'strategy'
        tcfd_none = 'none'
        tcfd_metrics = 'metrics'
        tcfd_risk = 'risk'
        tcfd_governance = 'governance'

        climate_sentiment_opportunity = 'opportunity'
        climate_sentiment_neutral = 'neutral'
        climate_sentiment_risk = 'risk'

        netzero_reduction_none = 'none'
        netzero_reduction_reduction = 'reduction'
        netzero_reduction_net_zero = 'net-zero'

        renewable_yes = 'LABEL_1'
        renewable_no = 'LABEL_0'

## ───────────────────────

## 1. Data

#### A. Load

In [ ]:
# | Configs
# |-------------------
with open(GLOBALS.paths.configs, 'r') as configs:
    configs = json.load(configs)
    configs = {(config['label'], config['year']): config for config in configs}

# | Workbooks
# |-------------------
workbooks = []

for year in tqdm(GLOBALS.inputs.years):
    for label in GLOBALS.inputs.labels:
        # Get the workbook data + config
        data = read_excel(GLOBALS.paths.raw.format(label=label, year=year))
        config = configs.get((label, year))

        # Create the workbook
        workbook = Workbook(
            data,
            year=config['year'],
            label=config['label'],
            join=config['join'],
            sheets=config['sheets'],
            merges=config['merges'],
            redundant=config['redundant'],
            drop=config['drop'],
            renames=config['renames']
        )
        workbooks.append(workbook)

#### B. Exploration

In [ ]:
# Filter workbooks and reinitialize
filtered = []
for workbook in workbooks:
    data = GLOBALS.models.processor._filter_workbook(workbook, workbook.sheets)
    workbook = Workbook(data, workbook.year, workbook.label, workbook.join)
    filtered.append(workbook)

# Find sheets and redundant columns
sheets = GLOBALS.models.eda.sheets(workbooks)
columns = GLOBALS.models.eda.columns(filtered)
redundant = GLOBALS.models.eda.redundant(filtered)

# Save results in parquet
write_csv(sheets, GLOBALS.paths.eda_sheets)
write_csv(columns, GLOBALS.paths.eda_columns)
write_csv(redundant, GLOBALS.paths.eda_redundant)

#### C. Optimization

In [ ]:
# | Mapping
# |-------------------
samples = GLOBALS.models.processor.process_workbooks(workbooks)

# | Manual
# |-------------------
mapping_truth = (
    read_csv(GLOBALS.paths.mapping_truth)
    .select(pl.col(f'{sample.year}_{sample.label}') for sample in samples)
    .filter(~pl.all_horizontal(pl.all().is_null()))
)

# | Optimization
# |-------------------
def evaluation_func(source, target):
    total = 0
    correct = 0

    for source_column_a, source_column_b in pairwise(source):
        target_column_a = target.get_column(source_column_a.name)
        target_column_b = target.get_column(source_column_b.name)

        source_mappings = dict(zip(source_column_a, source_column_b))
        target_mappings = dict(zip(target_column_a, target_column_b))

        for source_value_a, source_value_b in source_mappings.items():
            target_value_a = source_value_a
            target_value_b = target_mappings.get(target_value_a)

            if source_value_b and target_value_b and source_value_b == target_value_b:
                correct += 1

        total += len(source_mappings)

    return correct / total


def objective(trial: Trial):
    # Sample parameters from continuous and categorical distributions
    w_D_col = trial.suggest_float('w_D_col', 0.1, 2.0)
    w_S_col = trial.suggest_float('w_S_col', 0.1, 2.0)
    w_S_field = trial.suggest_float('w_S_field', 0.1, 2.0)
    w_S_context = trial.suggest_float('w_S_context', 0.1, 2.0)
    n_fields = trial.suggest_int('n_fields', 1, 7)
    win_S_context = trial.suggest_int('win_S_context', 1, 7)
    thresh_C_method = trial.suggest_categorical('thresh_C_method', ['fixed', 'cluster', 'spread'])
    thresh_C = trial.suggest_float('thresh_C', 0.1, 2.0)
    thresh_C_z_factor = trial.suggest_float('thresh_C_z_factor', 0.1, 2.0)
    
    # Update merger with the new parameters
    merger = Merger(
        GLOBALS.models.similarity,
        w_D_col=w_D_col,
        w_S_col=w_S_col,
        w_S_field=w_S_field,
        w_S_context=w_S_context,
        n_fields=n_fields,
        win_S_context=win_S_context,
        thresh_C_method=thresh_C_method,
        thresh_C=thresh_C,
        thresh_C_z_factor=thresh_C_z_factor
    )

    # Run the merge process (this returns merged DataFrame and mapping).
    mapping = merger.map(samples)

    # Evaluate the merged result.
    score = evaluation_func(mapping, mapping_truth)
    
    # Optuna will maximize the score.
    return score

# Create an Optuna study object and optimize
study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=200)

print(f'Best Parameters: {study.best_params}\nBest Score: {study.best_value}')

## 2. Processing

#### A. Run

In [ ]:
processed = GLOBALS.models.processor.process_workbooks(workbooks)

#### B. Write

In [ ]:
for df in processed:
    write_parquet(df, GLOBALS.paths.processed.format(label=df.label, year=df.year))

## 3. Merging

#### X. Manual

In [ ]:
# | 1. Collect data
# |-------------------

dfs = []
for workbook in tqdm(workbooks, desc='Processing workbooks'):
    wb = GLOBALS.models.processor._filter_workbook(workbook, workbook.sheets)
    df = GLOBALS.models.processor._join_workbook(wb, workbook.join)
    df = GLOBALS.models.processor._clean_fields(df)
    df = GLOBALS.models.processor._drop_columns(df, [])

    df = DataFrame(df, year=workbook.year, label=workbook.label)
    dfs.append(df)

In [ ]:
# | 2. Create mapping
# |-------------------

mapping = (
    pl.read_csv(GLOBALS.paths.mapping)

    # Combine column and sheet values
    .with_columns(
        pl.concat_str(
            pl.col(f'{year}__{label}__column'),
            pl.col(f'{year}__{label}__sheet'),
            separator='__'
        ).alias(f'{year}_{label}')
        for year in GLOBALS.inputs.years for label in GLOBALS.inputs.labels
    )

    # Only keep new combined columns
    .select(
        pl.col(f'{year}_{label}')
        for year in GLOBALS.inputs.years for label in GLOBALS.inputs.labels
    )

    # Keep rows that don't have all values null
    .filter(~pl.all_horizontal(pl.all().is_null()))

    # Convert to standardized format
    .with_columns(
        pl.all().map_elements(
            lambda col: GLOBALS.models.processor.create_column(
                {'column': col.split('__')[0], 'sheet': col.split('__')[1], 'value': 'response'},
            ),
            return_dtype=pl.String
        )
    )
)

# Verify that all mapping columns contain unique values
for col in mapping:
    clean = col.drop_nulls()
    if len(clean.unique()) != len(clean):
        counts = clean.value_counts().filter(pl.col('count') > 1)
        print(f'ERROR: {col.name} contains non-unique values:')
        print(counts)

# Coalesce the mapping
mapping_coalesced = (
    mapping
    .select(mapping.columns[::-1])
    .with_columns(pl.coalesce(pl.all()).alias('column'))
    .select(pl.col('column'))
    .to_series()
)

# Make the columns unique by adding number properties
mapping_coalesced = (
    pl.DataFrame(mapping_coalesced)
    .with_row_index('order')
    .group_by('column')
    .agg(pl.len().alias('number'), 'order')
    .with_columns(pl.col('order').list.first())
    .with_columns(pl.int_ranges(0, pl.col('number')).alias('number'))
    .explode('number')
    .sort('order')
    .map_rows(lambda x: GLOBALS.models.processor.update_column(x[0], {'number': str(x[1])}) if x[1] > 0 else x[0], return_dtype=pl.String)
    .get_column('map')
)

In [ ]:
# | 3. Merge data
# |-------------------

# Define metadata columns for the output dataframe
year_column = 'column=year|sheet=*|value=meta'
label_column = 'column=label|sheet=*|value=meta'

# Vertical stack the dataframes
merged = []

for df in dfs:
    mapping_column = mapping.get_column(df.name)

    # Get mapping from old to new columns
    column_mapping = {old: new for old, new in zip(mapping_column, mapping_coalesced) if old}

    # Find columns in column mapping that aren't in the dataframe
    for col in column_mapping.keys():
        if col not in df.columns:
            print(f'{df.name} — ERROR: `{col}` not found')

    # Update dataframe
    df = (
        df
        # Select columns from mapping column, since the mapping column leaves out some columns
        .select(column_mapping.keys())

        # Rename the columns
        .rename(column_mapping)

        # Add meta columns
        .with_columns(
            pl.lit(df.year).alias(year_column),
            pl.lit(df.label).alias(label_column)
        )

        # Reorder columns
        .select(year_column, label_column, pl.exclude(year_column, label_column))
    )

    merged.append(df)

# Cast to standard type
merged = pl.concat(merged, how='diagonal_relaxed').cast(pl.List(pl.String))

#### A. Load

In [ ]:
processed = read_parquet(GLOBALS.paths.processed)

#### B. Run

In [ ]:
merged, mapping = GLOBALS.models.merger.merge(processed)

#### C. Write

In [ ]:
write_parquet(merged, GLOBALS.paths.merged)
write_csv(mapping, GLOBALS.paths.mapping)

#### D. Coalesce

In [ ]:
# Coalesce column names in mapping
coalesced_rows = []
for row in list(mapping.iter_rows(named=True)):
    coalesced_row = {}
    row_items = list(row.items())[::-1]

    for i, (k, v) in enumerate(row_items):
        if v and not coalesced_row:
            coalesced_row['column'] = v
            coalesced_row['to'] = k

        elif (not v and coalesced_row) or (i == len(row_items) - 1 and coalesced_row):
            coalesced_row['from'] = row_items[i - 1][0]
            break
    
    coalesced_rows.append(coalesced_row)

mapping_coalesced = pl.DataFrame(coalesced_rows)

# Write coalesced mapping to dataframe
write_csv(mapping_coalesced, GLOBALS.paths.mapping_coalesced)

## ───────────────────────

## 4. Columns

#### A. Load

In [ ]:
merged = read_parquet(GLOBALS.paths.merged)

#### B. Filter

In [ ]:
frq = defaultdict(list)
year = 'column=year|sheet=*|value=meta'

for col in merged.columns:

    # Skip the year column
    if col in [year]:
        continue

    # Get the groups by year
    groups = (
        merged
        .select(year, col)
        .with_columns(pl.col(year).list.first())
        .explode(col)
        .drop_nulls(col)
        .group_by(year)
    )

    # Get the values for the column
    values = merged.get_column(col).drop_nulls().explode()

    # Check if there are values
    if values.is_empty():
        frq['column'].append(None)
        frq['number values'].append(None)
        frq['number unique values'].append(None)
        frq['average number unique values across years'].append(None)
        frq['proportion unique'].append(None)
        frq['average length'].append(None)
        frq['length range'].append(None)
        frq['punctuation contained'].append(None)
        frq['values contain other number'].append(None)
        frq['samples'].append(None)
        continue

    # Check proportion of unique values to values
    number_values = values.len()
    number_unique_values = values.unique().len()
    average_number_unique_values_across_years = groups.agg(pl.col(col).unique().len()).mean().get_column(col).item()
    proportion_unique = values.unique().len() / values.len()
    average_length = values.str.len_chars().mean()
    length_range = values.str.len_chars().min(), values.str.len_chars().max()
    punctuation_contained = pl.Series(regex.findall(r'\p{P}', values.str.concat('').item())).unique().drop_nulls().to_list() or None
    values_contain_other_number = sum(values.str.count_matches(p).sum() for p in [r'^Other:\s', r'^Other,\s'])
    samples = values.value_counts(sort=True, name='count')
    samples = pl.concat((samples.slice(0, 3), samples.slice(-3, 3)), how='vertical')
    samples = samples.with_columns(pl.concat_str(pl.col('count'), pl.col(col), separator='|').alias('samples')).get_column('samples')
    samples = samples.str.slice(0, 30).to_list()

    # Append responses to the dataframe
    frq['column'].append(col)
    frq['number values'].append(number_values)
    frq['number unique values'].append(number_unique_values)
    frq['average number unique values across years'].append(average_number_unique_values_across_years)
    frq['proportion unique'].append(proportion_unique)
    frq['average length'].append(average_length)
    frq['length range'].append(length_range)
    frq['punctuation contained'].append(punctuation_contained)
    frq['values contain other number'].append(values_contain_other_number)
    frq['samples'].append(samples)

frq = pl.DataFrame(frq)

In [ ]:
filtered = (
    frq
    .filter(
        (pl.col('values contain other number') > 0) |
        (
            (pl.col('punctuation contained').list.len() > 0) &
            (pl.col('punctuation contained').list.contains('.')) &
            (pl.col('punctuation contained').list.contains(',')) &
            (pl.col('average length') > 20) &
            (pl.col('average number unique values across years') > 1) &
            (pl.col('length range').list.last() > 30)
        )
    )
    .with_columns(pl.col(pl.List(pl.String)).list.join('    ·    '))
)

RESPONSE_COLUMNS = GLOBALS.models.processor.filter_columns(merged.columns, {'value': 'response'})

FREE_RESPONSE_COLUMNS = filtered.get_column('column').to_list()

## ───────────────────────

## 5. Translation

#### A. Load

In [ ]:
merged = read_parquet(GLOBALS.paths.merged)
translate = merged.select(merged.columns) # FREE_RESPONSE_COLUMNS

#### B. Run

In [ ]:
translated = GLOBALS.models.translator.translate_df(translate, processor=GLOBALS.models.processor, extends=False, detected=False, progress=True, batch_size=64)

In [ ]:
translated = merged.with_columns(translated)

#### C. Write

In [ ]:
write_parquet(translated, GLOBALS.paths.translated)

## 6. Classification

#### A. Load

In [ ]:
translated = read_parquet(GLOBALS.paths.translated)
classify = translated.select(translated.columns) # FREE_RESPONSE_COLUMNS

#### B. Run

In [ ]:
classified = GLOBALS.models.classifier.classify_df(classify, processor=GLOBALS.models.processor, extends=True, batch_size=512, progress=True, chunk_path=GLOBALS.paths.classified_chunk, chunk_makedirs=False)

In [ ]:
classified = translated.with_columns(classified)

#### C. Write

In [ ]:
write_parquet(classified, GLOBALS.paths.classified)

## 7. Metrics

#### A. Load

In [ ]:
classified = read_parquet(GLOBALS.paths.classified)

#### B. Run

In [ ]:
columns = []

# Define important function aliases
update_column = GLOBALS.models.processor.update_column
create_column = GLOBALS.models.processor.create_column

# Create expressions for each response column
for col in GLOBALS.models.processor.filter_columns(classified.columns, {'value': 'response'}):

    # Define original and new column names
    specificity_column_name = update_column(col, {'model': 'climate_specificity', 'label': GLOBALS.labels.climate_specificity_yes, 'value': 'score'})
    average_specificity_column_name = update_column(col, {'model': 'climate_specificity', 'label': GLOBALS.labels.climate_specificity_yes, 'value': 'score_average'})
    response_length_column_name = update_column(col, {'value': 'number_characters'})
    number_responses_column_name = update_column(col, {'value': 'number_responses'})

    # Create expressions
    average_specificity_column = (
        pl.col(specificity_column_name)
        .list.drop_nulls()
        .cast(pl.List(pl.Float64))
        .list.mean()
        .alias(average_specificity_column_name)
    )
    response_length_column = (
        pl.col(col)
        .list.drop_nulls()
        .cast(pl.List(pl.String))
        .list.join('', ignore_nulls=True)
        .str.len_chars()
        .alias(response_length_column_name)
    )
    number_responses_column = (
        pl.col(col)
        .list.drop_nulls()
        .list.len()
        .alias(number_responses_column_name)
    )

    # Add column expressions
    columns.extend((average_specificity_column, response_length_column, number_responses_column))

# Add expressions
metrics = classified.with_columns(columns)

# # Add aggregate metrics
# response_columns_free_specificity_scores = [GLOBALS.models.processor.update_column(col, {'model': 'climate_specificity', 'label': GLOBALS.labels.climate_specificity_yes, 'value': 'score'}) for col in RESPONSE_COLUMNS_FREE]
# response_columns_free_num_chars = [GLOBALS.models.processor.update_column(col, {'value': 'num_chars'}) for col in RESPONSE_COLUMNS_FREE]
# response_columns_num_responses = [GLOBALS.models.processor.update_column(col, {'value': 'num_responses'}) for col in RESPONSE_COLUMNS]

# response_columns_free_specificity_scores_average_name = GLOBALS.models.processor.create_column({'column': 'free_response', 'sheet': GLOBALS.models.processor.column_suffix_special, 'model': 'climate_specificity', 'label': GLOBALS.labels.climate_specificity_yes, 'value': 'average_score'})
# response_columns_free_num_chars_total_name = GLOBALS.models.processor.create_column({'column': 'free_response', 'sheet': GLOBALS.models.processor.column_suffix_special, 'value': 'num_chars'})
# response_columns_num_responses_total_name = GLOBALS.models.processor.create_column({'column': 'any_response', 'sheet': GLOBALS.models.processor.column_suffix_special, 'value': 'num_responses'})
# response_columns_num_questions_responded_total_name = GLOBALS.models.processor.create_column({'column': 'any_response', 'sheet': GLOBALS.models.processor.column_suffix_special, 'value': 'num_questions_responded'})

# metrics = metrics.with_columns(
#     pl.concat_list(response_columns_free_specificity_scores).list.drop_nulls().cast(pl.List(pl.Float64)).list.mean().alias(response_columns_free_specificity_scores_average_name),
#     pl.concat_list(response_columns_free_num_chars).list.sum().alias(response_columns_free_num_chars_total_name),
#     pl.concat_list(response_columns_num_responses).list.sum().alias(response_columns_num_responses_total_name),
#     pl.concat_list(response_columns_num_responses).list.eval(pl.when(pl.element() >= 1).then(1).otherwise(0)).list.sum().alias(response_columns_num_questions_responded_total_name),
# )

#### C. Write

In [ ]:
write_parquet(metrics.with_columns(merged), GLOBALS.paths.metrics)

## ───────────────────────

## 8. Analysis

#### A. Load

In [7]:
metrics = read_parquet(GLOBALS.paths.metrics)

In [8]:
# Only consider investor responses to avoid double counting
metrics = metrics.filter(pl.col('column=label|sheet=*|value=meta').list.contains('investor'))

# Clean and modify columns of the dataframe
columns = []
for column in metrics.select(pl.selectors.by_dtype(pl.List(pl.String))):

    # Turn single element list columns into string columns
    if column.drop_nulls().list.len().eq(1).all():
        column = column.explode().cast(pl.String)

    # Turn string columns that contain into float columns
    if column.dtype.base_type().is_(pl.String):
        column_casted = column.cast(pl.Float64, strict=False)

    # Turn all list columns that contain strings into numeric
    elif column.dtype.is_(pl.List(pl.String)):
        column_casted = column.cast(pl.List(pl.Float64), strict=False)

    # Check if the cast was successful, then replace the original column
    if column.explode().null_count() == column_casted.explode().null_count():
        column = column_casted

    # Append final transformed column
    columns.append(column)

# Replace original columns with transformed columns
metrics = metrics.with_columns(columns)

#### B. Constants

In [ ]:
# | Columns
# |-------------------
LABEL_COLUMN = 'column=label|sheet=*|value=meta'
YEAR_COLUMN = 'column=year|sheet=*|value=meta'
ACCOUNT_NUMBER_COLUMN = 'column=Account number|sheet=*|value=response'
CURRENCY_COLUMN = 'column=C0.4_Select the currency used for all financial information disclosed throughout your response.|sheet=C0 - Introduction|value=response'
PAYBACK_PERIOD_COLUMN = 'column=C4.3b_C7_Provide details on the initiatives implemented in the reporting year in the table below. - Payback period|sheet=C4.3b|value=response'
INITIATIVES_INVESTMENT_COLUMN = 'column=C4.3b_C6_Provide details on the initiatives implemented in the reporting year in the table below. - Investment required (unit currency – as specified in C0.4)|sheet=C4.3b|value=response'
INTENSITY_TARGET_FIGURE_COLUMN_PER_CURRENCY = 'column=CC12.2 C1 - Please describe your gross global combined Scope 1 and 2 emissions for the reporting year in metric tonnes CO2e per unit currency total revenue - Intensity figure =|sheet=CC12.2|value=response'
INTENSITY_TARGET_FIGURE_COLUMN_PER_ACTIVITY = 'column=C4.1b_C14_Provide details of your emissions intensity target(s) and progress made against those target(s). - Intensity figure in reporting year (metric tons CO2e per unit of activity)|sheet=C4.1b|value=response'
INTENSITY_TARGET_PERCENT_ACHIEVED_COLUMN = 'column=C4.1b_C15_Provide details of your emissions intensity target(s) and progress made against those target(s). - % of target achieved [auto-calculated]|sheet=C4.1b|value=response'
TARGETED_REDUCTION_COLUMN = 'column=C4.1b_C10_Provide details of your emissions intensity target(s) and progress made against those target(s). - Targeted reduction from base year (%)|sheet=C4.1b|value=response'
YEAR_TARGET_SET_COLUMN = 'column=C4.1b_C2_Provide details of your emissions intensity target(s) and progress made against those target(s). - Year target was set|sheet=C4.1b|value=response'

# | Mappings
# |-------------------
# Mapping to convert currency codes to standardized format
CURRENCY_RENAMES = {
    'XXX'       : 'USD',
    'USN'       : 'USD',
    'EUR(€)'    : 'EUR',
    'GBP(£)'    : 'GBP',
    'JPY(¥)'    : 'JPY',
    'CAD ($)'   : 'CAD',
    'AUD ($)'   : 'AUD',
    'NZD ($)'   : 'NZD',
    'CNY(¥)'    : 'CNY',
    'INR(Rp)'   : 'INR',
    'SGD ($)'   : 'SGD',
    'ARS($)'    : 'ARS',
    'BRL(R$)'   : 'BRL',
    'MXN ($)'   : 'MXN',
    'ZAR (R)'   : 'ZAR',
    'USD($)'    : 'USD',
    'CHW'       : 'CHF',
}

# Historical exchange rates 2010 to 2020 IMF World Bank major currencies
EXCHANGE_RATES_PATH = '/Users/chase/Desktop/code/projects/mse/data/cdp/output/exchange_rates.csv'
EXCHANGE_RATES = read_csv(EXCHANGE_RATES_PATH).rows_by_key(key='year', named=True, unique=True)

#### C. Initiatives Investment vs. Payback Period

##### I. Standardize Currencies

In [14]:
converted = (
    metrics

    # Select columns of interest
    .select(
        ACCOUNT_NUMBER_COLUMN,
        YEAR_COLUMN,
        CURRENCY_COLUMN,
        PAYBACK_PERIOD_COLUMN,
        INITIATIVES_INVESTMENT_COLUMN
    )

    # Drop rows where the number of payback periods is not equal to the number of investments
    .filter(pl.col(PAYBACK_PERIOD_COLUMN).list.len() == pl.col(INITIATIVES_INVESTMENT_COLUMN).list.len())

    # Explode list fields
    .explode(PAYBACK_PERIOD_COLUMN, INITIATIVES_INVESTMENT_COLUMN)

    # Only select company rows where all three are provided
    .filter(
        pl.col(CURRENCY_COLUMN).is_not_null() &
        pl.col(PAYBACK_PERIOD_COLUMN).is_not_null() &
        pl.col(INITIATIVES_INVESTMENT_COLUMN).is_not_null()
    )

    # Drop rows where initiatives investment is not a value
    .filter(~pl.col(INITIATIVES_INVESTMENT_COLUMN).str.to_lowercase().is_in(['hidden answer', 'question not applicable']))

    # Drop payback period rows where question is not applicable
    .filter(~pl.col(PAYBACK_PERIOD_COLUMN).str.to_lowercase().is_in(['hidden answer', 'question not applicable']))

    # Convert initiatives investment to float
    .with_columns(pl.col(INITIATIVES_INVESTMENT_COLUMN).cast(pl.Float64, strict=True))

    # Drop initiatives where the amount invested is zero
    .filter(pl.col(INITIATIVES_INVESTMENT_COLUMN) > 0)

    # Rename currencies to standardized
    .with_columns(pl.col(CURRENCY_COLUMN).replace(CURRENCY_RENAMES))

    # Correct mistakes in disclosures
    .with_columns(pl.when((pl.col(ACCOUNT_NUMBER_COLUMN) == 20893) & (pl.col(YEAR_COLUMN) == 2017)).then(pl.lit('JPY')).otherwise(pl.col(CURRENCY_COLUMN)).alias(CURRENCY_COLUMN))

    # Convert currencies to USD
    .with_columns(
        pl.struct(YEAR_COLUMN, CURRENCY_COLUMN, INITIATIVES_INVESTMENT_COLUMN)
        .map_elements(
            lambda struct:
            struct[INITIATIVES_INVESTMENT_COLUMN] / EXCHANGE_RATES[struct[YEAR_COLUMN]][struct[CURRENCY_COLUMN]],
            return_dtype=pl.Float64
        )
        .alias(INITIATIVES_INVESTMENT_COLUMN)
    )

    # Create our own categories: <1 year, 1-3 years, >3 years
    .with_columns(
        pl.col(PAYBACK_PERIOD_COLUMN)
        .replace(['4-10 years', '4 - 10 years', '11-15 years', '21-25 years', '>25 years', '16-20 years'], '>3 years')
        .replace(None, 'No payback')
    )
)

# converted

##### II. Sample Investment Amounts vs. Time

In [ ]:
start, end = 90, 100

selected = (
    # Group by year and account number
    converted
    .select(YEAR_COLUMN, ACCOUNT_NUMBER_COLUMN, INITIATIVES_INVESTMENT_COLUMN)
    .group_by(ACCOUNT_NUMBER_COLUMN, YEAR_COLUMN)
    .agg(pl.col(INITIATIVES_INVESTMENT_COLUMN).sum())
    .sort(by=ACCOUNT_NUMBER_COLUMN)

    # Select range of companies from start to end
    .filter(
        pl.col(ACCOUNT_NUMBER_COLUMN).is_in(
            converted
            .select(ACCOUNT_NUMBER_COLUMN).unique(maintain_order=True)
            .slice(start, end - start).get_column(ACCOUNT_NUMBER_COLUMN)
        )
    )

    # Rename columns for clarity
    .rename({
        YEAR_COLUMN: 'Year',
        ACCOUNT_NUMBER_COLUMN: 'Account',
        INITIATIVES_INVESTMENT_COLUMN: 'Total Investment (USD)'
    })
)

alt.Chart(selected).mark_line().encode(
    x=alt.X(shorthand='Year', scale=alt.Scale(domain=[2010, 2020])),
    y=alt.Y(shorthand='Total Investment (USD)'),
    color='Account:N'
).interactive()

alt.Chart(...)

##### III. Comprehensive Analysis

In [ ]:
# Rename the columns for easier access
df = converted.rename({
    PAYBACK_PERIOD_COLUMN: "payback_period",
    INITIATIVES_INVESTMENT_COLUMN: "investment_required"
})

# Set up a better figure size and style
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (20, 16)
plt.rcParams['font.size'] = 12

# Create a formatter for axis labels (to display large numbers in K, M format)
def millions_formatter(x, pos):
    if x == 0:
        return '0'
    elif abs(x) >= 1e6:
        return f'${x*1e-6:.1f}M'
    elif abs(x) >= 1e3:
        return f'${x*1e-3:.1f}K'
    else:
        return f'${x:.1f}'

formatter = FuncFormatter(millions_formatter)

# Define order for payback periods for consistent plotting
payback_order = ['<1 year', '1-3 years', '>3 years', 'No payback']

# Create a grid for our visualizations
fig = plt.figure(figsize=(20, 25))
gs = gridspec.GridSpec(5, 2, figure=fig, hspace=0.4, wspace=0.3)

# 1. Distribution of Initiatives by Year
ax1 = fig.add_subplot(gs[0, 0])
year_counts = df.group_by(YEAR_COLUMN).agg(pl.len().alias('count')).sort(YEAR_COLUMN)
year_counts_pd = year_counts.to_pandas()
# Updated barplot syntax
sns.barplot(data=year_counts_pd, x=YEAR_COLUMN, y='count', ax=ax1, hue=YEAR_COLUMN, palette='viridis', legend=False)
ax1.set_title('Number of Initiatives by Year', fontsize=16)
ax1.set_xlabel('Year', fontsize=14)
ax1.set_ylabel('Number of Initiatives', fontsize=14)
ax1.tick_params(axis='x', rotation=0)

# 2. Distribution of Payback Periods
ax2 = fig.add_subplot(gs[0, 1])
payback_counts = df.group_by('payback_period').agg(pl.len().alias('count'))
# Convert to pandas for categorical ordering
payback_counts_pd = payback_counts.to_pandas()
payback_counts_pd['payback_period'] = pd.Categorical(
    payback_counts_pd['payback_period'], 
    categories=payback_order, 
    ordered=True
)
payback_counts_pd = payback_counts_pd.sort_values('payback_period')
# Updated barplot syntax
sns.barplot(data=payback_counts_pd, x='payback_period', y='count', ax=ax2, hue='payback_period', palette='viridis', legend=False)
ax2.set_title('Distribution of Payback Periods', fontsize=16)
ax2.set_xlabel('Payback Period', fontsize=14)
ax2.set_ylabel('Number of Initiatives', fontsize=14)
ax2.tick_params(axis='x', rotation=45)

# 3. Distribution of Investment Amounts (Histogram)
ax3 = fig.add_subplot(gs[1, 0])
# Filter in Polars then convert only what's needed for plotting
investment_data = df.filter(pl.col('investment_required') > 0).select('investment_required').to_pandas()
sns.histplot(investment_data, ax=ax3, bins=30, kde=True, log_scale=True)
ax3.set_title('Distribution of Investment Amounts (Log Scale)', fontsize=16)
ax3.set_xlabel('Investment Required (USD, Log Scale)', fontsize=14)
ax3.set_ylabel('Frequency', fontsize=14)
ax3.xaxis.set_major_formatter(formatter)

# 4. Box Plot of Investment by Payback Period
ax4 = fig.add_subplot(gs[1, 1])
# Calculate 99th percentile in Polars
investment_threshold = df.filter(pl.col('investment_required') > 0).select(
    pl.col('investment_required').quantile(0.99)
).item()
# Filter data in Polars
filtered_df = df.filter(
    (pl.col('investment_required') > 0) & 
    (pl.col('investment_required') <= investment_threshold)
)
# Convert to pandas for categorical ordering
filtered_pd = filtered_df.to_pandas()
filtered_pd['payback_period'] = pd.Categorical(
    filtered_pd['payback_period'], 
    categories=payback_order, 
    ordered=True
)
# Updated boxplot syntax
sns.boxplot(data=filtered_pd, x='payback_period', y='investment_required', ax=ax4, hue='payback_period', palette='viridis', legend=False)
ax4.set_title('Investment Amount by Payback Period (Excluding Outliers)', fontsize=16)
ax4.set_xlabel('Payback Period', fontsize=14)
ax4.set_ylabel('Investment Required (USD)', fontsize=14)
ax4.yaxis.set_major_formatter(formatter)
ax4.tick_params(axis='x', rotation=45)

# 5. Trend of Investments Over Years (Mean and Median)
ax5 = fig.add_subplot(gs[2, 0])
yearly_investment = df.filter(pl.col('investment_required') > 0).group_by(YEAR_COLUMN).agg(
    pl.col('investment_required').mean().alias('mean_investment'),
    pl.col('investment_required').median().alias('median_investment')
).sort(YEAR_COLUMN)
yearly_investment_pd = yearly_investment.to_pandas()
ax5.plot(yearly_investment_pd[YEAR_COLUMN], yearly_investment_pd['mean_investment'], marker='o', label='Mean Investment')
ax5.plot(yearly_investment_pd[YEAR_COLUMN], yearly_investment_pd['median_investment'], marker='s', label='Median Investment')
ax5.set_title('Average Investment Amount by Year', fontsize=16)
ax5.set_xlabel('Year', fontsize=14)
ax5.set_ylabel('Investment Amount (USD)', fontsize=14)
ax5.yaxis.set_major_formatter(formatter)
ax5.legend()
ax5.grid(True)

# 6. Stacked Bar Chart - Payback Period Distribution by Year
ax6 = fig.add_subplot(gs[2, 1])
payback_by_year = df.group_by([YEAR_COLUMN, 'payback_period']).agg(pl.len().alias('count')).to_pandas()
# Pivot the data for stacked bar chart
payback_pivot = payback_by_year.pivot(index=YEAR_COLUMN, columns='payback_period', values='count').fillna(0)
# Reorder columns
payback_pivot = payback_pivot.reindex(columns=payback_order)
payback_pivot.plot(kind='bar', stacked=True, ax=ax6, colormap='viridis')
ax6.set_title('Distribution of Payback Periods by Year', fontsize=16)
ax6.set_xlabel('Year', fontsize=14)
ax6.set_ylabel('Number of Initiatives', fontsize=14)
ax6.legend(title='Payback Period')

# 7. Heatmap - Correlation between Year, Payback Period, and Investment
# Convert payback period to numeric for correlation
payback_map = {'<1 year': 1, '1-3 years': 2, '>3 years': 3, 'No payback': 4}
# Create a temporary pandas DF for correlation
correlation_data_pd = df.to_pandas()
correlation_data_pd['payback_numeric'] = correlation_data_pd['payback_period'].map(payback_map)
correlation_matrix = correlation_data_pd[[YEAR_COLUMN, 'payback_numeric', 'investment_required']].corr()

ax7 = fig.add_subplot(gs[3, 0])
sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', ax=ax7)
ax7.set_title('Correlation Matrix', fontsize=16)

# 8. Scatter Plot: Investment Amount vs. Payback Period
ax8 = fig.add_subplot(gs[3, 1])
sns.scatterplot(data=filtered_pd, x='investment_required', y='payback_period', 
                hue=YEAR_COLUMN, palette='viridis', alpha=0.7, ax=ax8)
ax8.set_title('Investment Amount vs. Payback Period', fontsize=16)
ax8.set_xlabel('Investment Required (USD)', fontsize=14)
ax8.set_ylabel('Payback Period', fontsize=14)
ax8.xaxis.set_major_formatter(formatter)
ax8.legend(title='Year')

# 9. Average Investment by Payback Period and Year
investment_by_payback_year = df.filter(pl.col('investment_required') > 0).group_by([YEAR_COLUMN, 'payback_period']).agg(
    pl.col('investment_required').mean().alias('mean_investment')
).sort([YEAR_COLUMN, 'payback_period']).to_pandas()

# Create categorical order
investment_by_payback_year['payback_period'] = pd.Categorical(
    investment_by_payback_year['payback_period'], 
    categories=payback_order, 
    ordered=True
)

ax9 = fig.add_subplot(gs[4, 0:])
sns.barplot(data=investment_by_payback_year, x=YEAR_COLUMN, y='mean_investment', 
            hue='payback_period', palette='viridis', ax=ax9)
ax9.set_title('Average Investment by Payback Period and Year', fontsize=16)
ax9.set_xlabel('Year', fontsize=14)
ax9.set_ylabel('Mean Investment (USD)', fontsize=14)
ax9.yaxis.set_major_formatter(formatter)
ax9.legend(title='Payback Period')

# Handle tight_layout warning by using fig.tight_layout() instead
fig.tight_layout(pad=2.0, h_pad=3.0, w_pad=3.0)
plt.show()

# Time Series Decomposition plots
# Create a datetime for visualization purposes
df_time = df.with_columns(pl.col(YEAR_COLUMN).cast(pl.Date).alias('date'))
investment_time_series = df_time.group_by('date').agg(
    pl.col('investment_required').mean().alias('mean'),
    pl.col('investment_required').median().alias('median'),
    pl.len().alias('count')
).sort('date').to_pandas()

plt.figure(figsize=(15, 10))
plt.subplot(3, 1, 1)
plt.plot(investment_time_series['date'], investment_time_series['mean'], marker='o')
plt.title('Mean Investment Over Time')
plt.ylabel('Mean Investment (USD)')
plt.gca().yaxis.set_major_formatter(formatter)
plt.grid(True)

plt.subplot(3, 1, 2)
plt.plot(investment_time_series['date'], investment_time_series['median'], marker='o', color='orange')
plt.title('Median Investment Over Time')
plt.ylabel('Median Investment (USD)')
plt.gca().yaxis.set_major_formatter(formatter)
plt.grid(True)

plt.subplot(3, 1, 3)
plt.plot(investment_time_series['date'], investment_time_series['count'], marker='o', color='green')
plt.title('Number of Initiatives Over Time')
plt.ylabel('Count')
plt.grid(True)

plt.tight_layout(pad=2.0)
plt.show()

# Violin plot of investment by payback period
plt.figure(figsize=(15, 10))
# Updated violinplot syntax
sns.violinplot(data=filtered_pd, x='payback_period', y='investment_required', hue='payback_period', palette='viridis', legend=False)
plt.title('Distribution of Investment Amounts by Payback Period', fontsize=16)
plt.xlabel('Payback Period', fontsize=14)
plt.ylabel('Investment Required (USD)', fontsize=14)
plt.gca().yaxis.set_major_formatter(formatter)
plt.grid(True)
plt.show()

# Stacked area chart of investment proportion by payback period over time
investment_proportion = df.group_by([YEAR_COLUMN, 'payback_period']).agg(
    pl.col('investment_required').sum().alias('total_investment')
).to_pandas()

# Pivot to get years as rows and payback periods as columns
investment_pivot = investment_proportion.pivot(index=YEAR_COLUMN, columns='payback_period', values='total_investment').fillna(0)
investment_pivot = investment_pivot.reindex(columns=payback_order)

# Calculate proportions
investment_prop = investment_pivot.div(investment_pivot.sum(axis=1), axis=0)

# Plot
plt.figure(figsize=(15, 10))
investment_prop.plot(kind='area', stacked=True, colormap='viridis')
plt.title('Proportion of Total Investment by Payback Period Over Time', fontsize=16)
plt.xlabel('Year', fontsize=14)
plt.ylabel('Proportion of Investment', fontsize=14)
plt.grid(True)
plt.legend(title='Payback Period')
plt.show()

# Investment quartile analysis
# Create investment quartiles using Polars
quartile_labels = ['Q1 (Smallest)', 'Q2', 'Q3', 'Q4 (Largest)']
df_with_quartiles = df.with_columns(
    pl.when(pl.col('investment_required') <= pl.col('investment_required').quantile(0.25))
    .then(pl.lit(quartile_labels[0]))
    .when(pl.col('investment_required') <= pl.col('investment_required').quantile(0.5))
    .then(pl.lit(quartile_labels[1]))
    .when(pl.col('investment_required') <= pl.col('investment_required').quantile(0.75))
    .then(pl.lit(quartile_labels[2]))
    .otherwise(pl.lit(quartile_labels[3]))
    .alias('investment_quartile')
)

# Convert to pandas for visualization
payback_by_quartile = df_with_quartiles.to_pandas().groupby(['investment_quartile', 'payback_period']).size().unstack(fill_value=0)
payback_by_quartile_pct = payback_by_quartile.div(payback_by_quartile.sum(axis=1), axis=0) * 100

# Plot this distribution
plt.figure(figsize=(15, 10))
payback_by_quartile_pct.plot(kind='bar', stacked=True, colormap='viridis')
plt.title('Payback Period Distribution by Investment Size Quartile', fontsize=16)
plt.xlabel('Investment Size Quartile', fontsize=14)
plt.ylabel('Percentage of Initiatives', fontsize=14)
plt.legend(title='Payback Period')
plt.grid(True)
plt.show()

# Evolution of payback periods over time
years = sorted(df.select(pl.col(YEAR_COLUMN).unique()).to_series().to_list())
strategy_shifts = pd.DataFrame(index=years)

# Calculate metrics using Polars
for year in years:
    year_data = df.filter(pl.col(YEAR_COLUMN) == year)
    year_count = year_data.height
    
    # Average investment size
    strategy_shifts.loc[year, 'avg_investment'] = year_data.select(pl.col('investment_required').mean()).item()
    
    # Proportion of initiatives with each payback period
    for period in payback_order:
        period_count = year_data.filter(pl.col('payback_period') == period).height
        strategy_shifts.loc[year, f'pct_{period.replace(" ", "_")}'] = period_count / year_count * 100

# Plot evolution of investment strategies
plt.figure(figsize=(15, 10))
for period in payback_order:
    column_name = f'pct_{period.replace(" ", "_")}'
    plt.plot(strategy_shifts.index, strategy_shifts[column_name], marker='o', label=period)

plt.title('Evolution of Payback Period Distribution Over Time', fontsize=16)
plt.xlabel('Year', fontsize=14)
plt.ylabel('Percentage of Initiatives', fontsize=14)
plt.grid(True)
plt.legend(title='Payback Period')
plt.show()

#### D. Emissions and Commitments Performance vs. Time

#### E. Intensity vs. Time

In [ ]:
ABSOLUTE_TARGETED_REDUCTION_COLUMN = 

# Using the dataframe preparation as provided
df = (
    metrics
    .select(
        YEAR_COLUMN,
        ACCOUNT_NUMBER_COLUMN,
        TARGETED_REDUCTION_COLUMN,
        YEAR_TARGET_SET_COLUMN
    )
    .filter(
        pl.col(TARGETED_REDUCTION_COLUMN).is_not_null() &
        pl.col(YEAR_TARGET_SET_COLUMN).is_not_null()
    )
    .explode(TARGETED_REDUCTION_COLUMN, YEAR_TARGET_SET_COLUMN)
    .filter(
        ~pl.col(TARGETED_REDUCTION_COLUMN).str.to_lowercase().is_in(['question not applicable', 'hidden answer']) &
        ~pl.col(YEAR_TARGET_SET_COLUMN).str.to_lowercase().is_in(['question not applicable', 'hidden answer'])
    )
    .with_columns(
        pl.col(TARGETED_REDUCTION_COLUMN).cast(pl.Float64, strict=True),
        pl.col(YEAR_TARGET_SET_COLUMN).cast(pl.Float64, strict=True)
    )
)

# Group by the year the target was set to analyze ambition trends
yearly_ambition = (
    df.group_by(YEAR_TARGET_SET_COLUMN)
    .agg([
        pl.col(TARGETED_REDUCTION_COLUMN).mean().alias('avg_targeted_reduction'),
        pl.col(TARGETED_REDUCTION_COLUMN).median().alias('median_targeted_reduction'),
        pl.count().alias('count')
    ])
    .sort(YEAR_TARGET_SET_COLUMN)
)

# Create the plot
fig, ax = plt.subplots(figsize=(14, 8))

# Bar chart for the average targeted reduction
bars = ax.bar(yearly_ambition[YEAR_TARGET_SET_COLUMN], yearly_ambition['avg_targeted_reduction'], 
              color='teal', alpha=0.7, width=0.6)

# Add a trend line to highlight the pattern
x = yearly_ambition[YEAR_TARGET_SET_COLUMN]
y = yearly_ambition['avg_targeted_reduction']
z = np.polyfit(x, y, 1)
p = np.poly1d(z)
ax.plot(x, p(x), "r--", linewidth=2, label=f'Trend Line (Slope: {z[0]:.2f})')

# Add data labels showing the average reduction percentage
for i, bar in enumerate(bars):
    height = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2., height + 1,
            f'{yearly_ambition["avg_targeted_reduction"][i]:.1f}%',
            ha='center', va='bottom', fontsize=9)

# Add sample size annotations
for i, year in enumerate(yearly_ambition[YEAR_TARGET_SET_COLUMN]):
    ax.text(year, 3, f'n={yearly_ambition["count"][i]}', 
            ha='center', va='bottom', fontsize=9, rotation=90,
            bbox=dict(facecolor='white', alpha=0.7, edgecolor='none'))

# Add median markers
ax.scatter(yearly_ambition[YEAR_TARGET_SET_COLUMN], yearly_ambition['median_targeted_reduction'], 
           color='darkred', marker='D', s=50, label='Median Targeted Reduction')

# Set labels and title
ax.set_xlabel('Year Target Was Set', fontsize=12)
ax.set_ylabel('Targeted Reduction from Base Year (%)', fontsize=12)
ax.set_title('Trends in Emission Reduction Ambition Over Time', fontsize=14)

# Adjust y-axis to start from 0
ax.set_ylim(bottom=0)

# Ensure x-axis shows all years as integers
ax.set_xticks(yearly_ambition[YEAR_TARGET_SET_COLUMN])
ax.set_xticklabels([str(int(year)) for year in yearly_ambition[YEAR_TARGET_SET_COLUMN]])

# Add a horizontal reference line showing the average ambition across all years
overall_avg = yearly_ambition['avg_targeted_reduction'].mean()
ax.axhline(y=overall_avg, color='gray', linestyle='--', alpha=0.7, 
           label=f'Overall Average: {overall_avg:.1f}%')

# Add annotations explaining the interpretation
higher_text = "↑ Higher values = More ambitious targets"
lower_text = "↓ Lower values = Less ambitious targets"
ax.text(yearly_ambition[YEAR_TARGET_SET_COLUMN].min(), yearly_ambition['avg_targeted_reduction'].max() * 0.9,
        higher_text, fontsize=10, bbox=dict(facecolor='white', alpha=0.7))
ax.text(yearly_ambition[YEAR_TARGET_SET_COLUMN].min(), yearly_ambition['avg_targeted_reduction'].max() * 0.8,
        lower_text, fontsize=10, bbox=dict(facecolor='white', alpha=0.7))

# Add grid and legend
ax.grid(True, linestyle='--', alpha=0.3, axis='y')
ax.legend(loc='best', fontsize=10)

plt.tight_layout()
plt.show()